In [2]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.datasets import make_classification

# 设置随机种子以确保结果可重现
np.random.seed(42)

# 创建一个小样本数据集
# 生成分类数据
X, y = make_classification(n_samples=8, n_features=4, n_redundant=0, n_informative=4,
                          n_clusters_per_class=1, n_classes=4, random_state=42, hypercube=True)
# 将特征值转换为整数,方便手算
X = np.round(X).astype(int)

print(X)
print(y)

# 多维数据，已经无法绘制二维图像了

[[ 1 -1 -1 -2]
 [ 4 -2 -1 -2]
 [-1  0  0 -1]
 [-1  1  2  1]
 [-1  0  2  1]
 [ 1  0 -2 -2]
 [ 1  0  0 -1]
 [-1 -1 -1 -1]]
[2 1 0 0 3 2 1 0]


处理时需注意矩阵的型，如1*2矩阵跟2*1矩阵不能直接相减，但在numpy处理下会变成2*2矩阵，很难察觉

In [3]:
def calcSW(X,y):
    n_features = X.shape[1]
    unique_classes = np.unique(y)    
    S_W = np.zeros((n_features, n_features))
    
    for cls in unique_classes:
        # 获取当前类别的所有样本
        class_samples = X[y == cls]
        
        # 计算当前类别的均值向量
        class_mean = np.mean(class_samples, axis=0)
        
        # 中心化当前类别的数据
        centered_data = class_samples - class_mean
        
        # 计算当前类别的散度矩阵并累加
        S_W += centered_data.T @ centered_data
    
    return S_W

def calcSB(X,y):
    # 将数据按类别分组
    n_features = X.shape[1]
    unique_classes = np.unique(y)  

    # 中心化当前类别的数据
    mean_overall = np.mean(X, axis=0).reshape(unique_classes.shape[0],1)

    S_B = np.zeros((n_features, n_features))

    for cls in unique_classes:
        # 获取当前类别的所有样本
        class_samples = X[y == cls]
        
        # 计算当前类别的均值向量
        class_mean = np.mean(class_samples, axis=0).reshape(unique_classes.shape[0],1)
        # 计算当前类别的散度矩阵并累加
        S_B += class_samples.shape[0] * (class_mean - mean_overall).dot((class_mean - mean_overall).T)
    return S_B


In [4]:
print("sw")
print(calcSW(X,y))
print("sb")
print(calcSB(X,y))

sw
[[ 4.5        -3.         -1.5        -1.5       ]
 [-3.          4.5         3.5         3.        ]
 [-1.5         3.5         5.66666667  3.83333333]
 [-1.5         3.          3.83333333  3.16666667]]
sb
[[17.375      -4.875      -8.125      -8.875     ]
 [-4.875       1.375       2.125       2.375     ]
 [-8.125       2.125       9.20833333  8.29166667]
 [-8.875       2.375       8.29166667  7.70833333]]


In [5]:
# 手动计算LDA步骤（用于验证）
def manual_lda(X, y):
    # 将数据按类别分组
    class0 = X[y == 0]
    class1 = X[y == 1]
    
    # 计算每个类别的均值向量
    mean0 = np.mean(class0, axis=0)
    mean1 = np.mean(class1, axis=0)
    
    # 计算类内散度矩阵
    S_W = calcSW(X,y)

    # 计算类间散度矩阵
    S_B = calcSB(X,y)
    
    # 计算最优投影方向
    # S_B * w = lambda * S_W * w
    # S_W^(-1) * S_B * w = lambda * w，等价于求矩阵S_W^(-1) * S_B的特征值，lambda为特征根，w为特征向量
    eig_vals, eig_vecs = np.linalg.eig(np.linalg.inv(S_W).dot(S_B))
    
    # 选择最大特征值对应的特征向量
    # 化简后J(w)=lambda,所以此处取最大的特征根
    max_eig_idx = np.argmax(eig_vals)
    w = eig_vecs[:, max_eig_idx]
    
    return w

In [6]:
# 执行手动LDA计算
w_manual = manual_lda(X, y)

# 执行自动LDA计算
lda = LinearDiscriminantAnalysis()
lda.fit(X, y)
w_sklearn = lda.scalings_[:, 0]  # 获取投影向量

In [7]:
print(w_manual)
print(w_manual[0]/w_manual[1])
print(w_sklearn)
print(w_sklearn[0]/w_sklearn[1])

[-0.40719743 -0.61248277 -0.06885714  0.67402804]
0.664830834848741
[ 1.04697884  1.57480488  0.17704425 -1.73304899]
0.6648308348487404


将投影方向与sklearn计算结果对比